# 02 · Latin-hypercube screening on the local machine

Workflow 2: evaluate a space-filling design over the four pricing/fleet knobs,
with every sample running as a subprocess on this machine. The entire study is
declared in [`studies/lhs-local.yaml`](../studies/lhs-local.yaml) — simulator,
runner, search space, metric, and phases — and `polarisopt` does the rest.

The same study can be driven two ways:

```bash
# CLI (what you'd use on a cluster — see workflow 3)
polarisopt run studies/lhs-local.yaml
polarisopt status studies/lhs-local.yaml
```

or from Python, which is what we do here so the results land in the same
session we analyze them in.

In [ ]:
from pathlib import Path

STUDY = Path("../studies/lhs-local.yaml")
print(STUDY.read_text())

## Run the study

`run_study` executes every phase. It is resume-safe: if the workspace already
holds finished samples (e.g. you ran the CLI first), they are not re-evaluated.

One Python-API detail: the CLI loads entry-point plugins (like taxidemo's
`taxi` simulator) on startup; from Python you call `load_external_plugins()`
yourself before running a study that uses one.

In [ ]:
from polarisopt.studies.runner import run_study
from polarisopt.utils.plugins import load_external_plugins

load_external_plugins()
samples = run_study(STUDY)
print(f"{len(samples)} samples, statuses: {set(s.status for s in samples)}")

## Load results from the SampleStore

Every sample — inputs, status, metric vector, runtime, folder — is persisted in
a SQLite SampleStore under the study workspace.

In [ ]:
import os
import pandas as pd
from polarisopt.samples.store import SampleStore
from polarisopt.utils.paths import workspace_layout

layout = workspace_layout(os.path.expanduser("~/taxidemo-runs/lhs-local"))
store = SampleStore.open(layout["db"], "taxi-lhs")
raw = store.to_dataframe()

PARAMS = ["taxi_count", "base_fare", "cost_per_tile", "max_multiplier"]   # YAML order
METRICS = ["profit", "missed", "pick_up_time", "journeys_completed"]      # identity-metric keys
df = pd.concat(
    [
        raw[["id", "status", "runtime_s"]],
        pd.DataFrame(raw["inputs"].tolist(), columns=PARAMS, index=raw.index),
        pd.DataFrame(raw["metric"].tolist(), columns=METRICS, index=raw.index),
    ],
    axis=1,
)
df.head()

## Marginal response of profit to each knob

One scatter per input, colored by missed customers. Two regimes are easy to
spot: high prices lose customers outright (yellow points, low profit), and
small fleets miss demand while very large fleets burn operating cost.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, p in zip(axes, PARAMS):
    sc = ax.scatter(df[p], df["profit"], c=df["missed"], cmap="viridis")
    ax.set_xlabel(p)
axes[0].set_ylabel("profit")
fig.colorbar(sc, ax=axes, label="missed customers", pad=0.01);

## Best designs found by the screen

In [ ]:
df.sort_values("profit", ascending=False).head(10)[PARAMS + METRICS].round(2)

## A crude sensitivity ranking

Rank correlation between each input and profit. (For a proper screening design
use the `morris` design type instead of `lhs` — same YAML, different
`phases[0].design`.)

In [ ]:
df[PARAMS + ["profit"]].corr(method="spearman")["profit"].drop("profit").sort_values(key=abs, ascending=False)

## Next

An LHS screen maps the landscape but spends samples uniformly. Workflow 3
([03_bo_crossover.ipynb](03_bo_crossover.ipynb), submitted via the CLI on
LCRC Crossover) uses these kinds of evaluations to warm-start a Gaussian
process and then *concentrates* samples where profit is highest.